## ✅ Cell 1 — Install Libraries

In [1]:
!pip install -q timm grad-cam albumentations 
print('✅ Libraries installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 102.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 112.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is in

In [2]:
!pip install -q --upgrade --force-reinstall Pillow==10.4.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 78.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


## ✅ Cell 2 — Locate Your Datasets

In [3]:
import os

WORKING_DIR    = '/kaggle/working'
MODEL_DIR      = os.path.join(WORKING_DIR, 'models')
HISTORY_DIR    = os.path.join(WORKING_DIR, 'history')
RESULTS_DIR    = os.path.join(WORKING_DIR, 'results')
CHECKPOINT_DIR = os.path.join(WORKING_DIR, 'checkpoints')
for d in [MODEL_DIR, HISTORY_DIR, RESULTS_DIR, CHECKPOINT_DIR]:
    os.makedirs(d, exist_ok=True)

print('Available input datasets:')
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.replace('/kaggle/input', '').count(os.sep)
    if depth < 4:
        print('  ' * depth + os.path.basename(root) or root)

# ── EDIT THESE PATHS based on what you see above ─────────────────
# Path to your previous best_model.pth (from the zip you uploaded as a dataset)
PREV_MODEL_PATH = '/kaggle/input/datasets/angaddevgan/deepfake-best-model-v1/best_model.pth'  # ← EDIT

# Path to 140k dataset (same as before)
DATA_140K = '/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake'  # ← EDIT if different

# Path to Celeb-DF dataset (edit based on the dataset you added)
DATA_CELEBDF = '/kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset/Celeb_V2'  # ← EDIT — check structure after adding

print('\nVerify these exist:')
for p in [PREV_MODEL_PATH, DATA_140K, DATA_CELEBDF]:
    print(f'  {"✅" if os.path.exists(p) else "❌"} {p}')

Available input datasets:
input
  datasets
    pranabr0y
      celebdf-v2image-dataset
    xhlulu
      140k-real-and-fake-faces
    angaddevgan
      deepfake-best-model-v1

Verify these exist:
  ✅ /kaggle/input/datasets/angaddevgan/deepfake-best-model-v1/best_model.pth
  ✅ /kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake
  ✅ /kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset/Celeb_V2


## ⚠️ Cell 2b — Inspect Celeb-DF structure
Run this to see how the Celeb-DF dataset is organized, since it varies by upload. We need to find folders of real vs fake face images.

In [4]:
# Explore the Celeb-DF dataset structure
for root, dirs, files in os.walk(DATA_CELEBDF):
    depth = root.replace(DATA_CELEBDF, '').count(os.sep)
    if depth < 3:
        n_files = len(files)
        indent = '  ' * depth
        print(f'{indent}{os.path.basename(root) or root}/  ({n_files} files)')

print('\n→ After seeing this, update CELEBDF_REAL_DIR and CELEBDF_FAKE_DIR below')

Celeb_V2/  (0 files)
  Val/  (0 files)
    fake/  (5068 files)
    real/  (5036 files)
  Test/  (0 files)
    fake/  (5067 files)
    real/  (5036 files)
  Train/  (0 files)
    fake/  (40536 files)
    real/  (40288 files)

→ After seeing this, update CELEBDF_REAL_DIR and CELEBDF_FAKE_DIR below


In [5]:
DATA_CELEBDF = '/kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset/Celeb_V2'  # adjust based on exact path shown

for split in ['Train', 'Val', 'Test']:
    split_path = os.path.join(DATA_CELEBDF, split)
    if os.path.exists(split_path):
        print(f'{split}/')
        for sub in os.listdir(split_path):
            sub_path = os.path.join(split_path, sub)
            if os.path.isdir(sub_path):
                n = len(os.listdir(sub_path))
                print(f'  {sub}/ → {n:,} images')

Train/
  fake/ → 40,536 images
  real/ → 40,288 images
Val/
  fake/ → 5,068 images
  real/ → 5,036 images
Test/
  fake/ → 5,067 images
  real/ → 5,036 images


## ⚠️ Cell 2c — (Only if Celeb-DF has VIDEOS, not images)
Skip this cell if Cell 2b showed image files (.jpg/.png). Run this only if you see .mp4 files — it extracts face crops from videos using MTCNN.

In [6]:
import cv2, torch
from facenet_pytorch import MTCNN
from tqdm.auto import tqdm

DEVICE_TMP = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mtcnn_extract = MTCNN(image_size=224, margin=20, post_process=False, device=DEVICE_TMP)

EXTRACTED_DIR = '/kaggle/working/celebdf_extracted'
os.makedirs(f'{EXTRACTED_DIR}/real', exist_ok=True)
os.makedirs(f'{EXTRACTED_DIR}/fake', exist_ok=True)

def extract_faces(video_dir, out_dir, max_videos=300, frames_per_video=5):
    videos = [f for f in os.listdir(video_dir) if f.endswith('.mp4')][:max_videos]
    saved = 0
    for v in tqdm(videos, desc=f'Extracting from {os.path.basename(video_dir)}'):
        cap = cv2.VideoCapture(os.path.join(video_dir, v))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        step = max(1, total // frames_per_video)
        for i in range(frames_per_video):
            cap.set(cv2.CAP_PROP_POS_FRAMES, i*step)
            ret, frame = cap.read()
            if not ret: break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            from PIL import Image as PILImage
            face = mtcnn_extract(PILImage.fromarray(rgb))
            if face is not None:
                face_np = face.permute(1,2,0).byte().numpy()
                cv2.imwrite(f'{out_dir}/{v}_{i}.jpg', cv2.cvtColor(face_np, cv2.COLOR_RGB2BGR))
                saved += 1
    return saved

# ── EDIT: point to the actual video folders in your Celeb-DF dataset ──
n_real = extract_faces(os.path.join(DATA_CELEBDF, 'Celeb-real'), f'{EXTRACTED_DIR}/real')
n_fake = extract_faces(os.path.join(DATA_CELEBDF, 'Celeb-synthesis'), f'{EXTRACTED_DIR}/fake')

print(f'✅ Extracted {n_real} real + {n_fake} fake faces')

# Update paths to use extracted images
CELEBDF_REAL_DIR = f'{EXTRACTED_DIR}/real'
CELEBDF_FAKE_DIR = f'{EXTRACTED_DIR}/fake'

ModuleNotFoundError: No module named 'facenet_pytorch'

## ✅ Cell 3 — Config

In [ ]:
import torch

CFG = {
    'model_name' : 'efficientnet_b4',
    'img_size'   : 224,
    'num_classes': 2,

    # Fine-tuning settings — much gentler than original training
    'epochs'         : 8,
    'batch_size'     : 32,
    'lr'             : 2e-5,        # 10x smaller than original (2e-4)
    'weight_decay'   : 1e-4,
    'label_smoothing': 0.1,
    'mixup_alpha'    : 0.2,

    'patience': 3,
    'num_workers': 0,
    'seed': 42,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

HISTORY_FILE    = os.path.join(HISTORY_DIR,    'finetune_history.json')
BEST_MODEL_PATH = os.path.join(MODEL_DIR,      'best_model_v2.pth')
LAST_CKPT_PATH  = os.path.join(CHECKPOINT_DIR, 'last_checkpoint_v2.pth')

print(f"Device: {CFG['device']}")
if CFG['device']=='cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## ✅ Cell 4 — Imports & Seed

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, ConcatDataset
from torchvision import datasets, transforms
import timm, json, time, random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix, f1_score, accuracy_score
from tqdm.auto import tqdm
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(CFG['seed'])
DEVICE = torch.device(CFG['device'])
print('✅ Ready')

## ✅ Cell 5 — Dataset Classes & Augmentation

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_aug = A.Compose([
    A.Resize(CFG['img_size'], CFG['img_size']),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.4),
    A.HueSaturationValue(p=0.3),
    A.GaussianBlur(blur_limit=(3,7), p=0.2),
    A.GaussNoise(p=0.2),
    A.ImageCompression(quality_lower=60, quality_upper=100, p=0.3),
    A.CoarseDropout(max_holes=8, max_height=20, max_width=20, p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.3),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

val_aug = A.Compose([
    A.Resize(CFG['img_size'], CFG['img_size']),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

class DeepfakeDataset(Dataset):
    """For datasets with ImageFolder structure: root/real/*.jpg, root/fake/*.jpg"""
    def __init__(self, root_dir, transform=None):
        self.dataset      = datasets.ImageFolder(root_dir)
        self.transform    = transform
        self.classes      = self.dataset.classes
        self.class_to_idx = self.dataset.class_to_idx

    def __len__(self):
        return len(self.dataset)

    def compute_fft_features(self, img_tensor):
        gray      = img_tensor.mean(dim=0).numpy()
        fft_shift = np.fft.fftshift(np.fft.fft2(gray))
        magnitude = np.log(np.abs(fft_shift) + 1e-8)
        magnitude = (magnitude - magnitude.min()) / (magnitude.max() - magnitude.min() + 1e-8)
        return torch.tensor(magnitude, dtype=torch.float32).unsqueeze(0).repeat(3, 1, 1)

    def __getitem__(self, idx):
        path, label = self.dataset.samples[idx]
        try:
            img = np.array(Image.open(path).convert('RGB'))
        except Exception:
            img = np.zeros((CFG['img_size'], CFG['img_size'], 3), dtype=np.uint8)

        if self.transform:
            tensor = self.transform(image=img)['image']
        else:
            tensor = torch.tensor(img).permute(2,0,1).float() / 255.0

        return tensor, self.compute_fft_features(tensor), label


class FlatDeepfakeDataset(Dataset):
    """For Celeb-DF style: two separate flat folders of real/ and fake/ images, label assigned manually."""
    def __init__(self, real_dir, fake_dir, transform=None):
        self.transform = transform
        self.samples = []
        if os.path.exists(real_dir):
            for f in os.listdir(real_dir):
                self.samples.append((os.path.join(real_dir, f), 1))  # 1 = real
        if os.path.exists(fake_dir):
            for f in os.listdir(fake_dir):
                self.samples.append((os.path.join(fake_dir, f), 0))  # 0 = fake
        self.class_to_idx = {'fake': 0, 'real': 1}

    def __len__(self):
        return len(self.samples)

    def compute_fft_features(self, img_tensor):
        gray      = img_tensor.mean(dim=0).numpy()
        fft_shift = np.fft.fftshift(np.fft.fft2(gray))
        magnitude = np.log(np.abs(fft_shift) + 1e-8)
        magnitude = (magnitude - magnitude.min()) / (magnitude.max() - magnitude.min() + 1e-8)
        return torch.tensor(magnitude, dtype=torch.float32).unsqueeze(0).repeat(3, 1, 1)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = np.array(Image.open(path).convert('RGB'))
        except Exception:
            img = np.zeros((CFG['img_size'], CFG['img_size'], 3), dtype=np.uint8)

        if self.transform:
            tensor = self.transform(image=img)['image']
        else:
            tensor = torch.tensor(img).permute(2,0,1).float() / 255.0

        return tensor, self.compute_fft_features(tensor), label

print('✅ Dataset classes ready')

## ✅ Cell 6 — Build Combined Train/Val Sets (140k + Celeb-DF)

In [ ]:
# ── Celeb-DF (pre-split into Train/Val/Test) ──────────────────────
train_celebdf = DeepfakeDataset(os.path.join(DATA_CELEBDF, 'Train'), transform=train_aug)
val_celebdf   = DeepfakeDataset(os.path.join(DATA_CELEBDF, 'Val'),   transform=val_aug)
test_celebdf  = DeepfakeDataset(os.path.join(DATA_CELEBDF, 'Test'),  transform=val_aug)

print(f'CelebDF classes: {train_celebdf.class_to_idx}')

# ── 140k dataset (existing) ───────────────────────────────────────
train_140k = DeepfakeDataset(os.path.join(DATA_140K, 'train'), transform=train_aug)
val_140k   = DeepfakeDataset(os.path.join(DATA_140K, 'valid'), transform=val_aug)
test_140k  = DeepfakeDataset(os.path.join(DATA_140K, 'test'),  transform=val_aug)

# ── Combine both datasets ──────────────────────────────────────────
train_ds = ConcatDataset([train_140k, train_celebdf])
val_ds   = ConcatDataset([val_140k, val_celebdf])
test_ds  = ConcatDataset([test_140k, test_celebdf])

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=CFG['num_workers'])
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'])
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'])

print(f'✅ Combined datasets:')
print(f'   Train: {len(train_ds):,} (140k: {len(train_140k):,} + CelebDF: {len(train_celebdf):,})')
print(f'   Val  : {len(val_ds):,}')
print(f'   Test : {len(test_ds):,}')

## ✅ Cell 7 — Model Architecture (same as before)

In [ ]:
class DualBranchDeepfakeDetector(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()
        self.rgb_backbone = timm.create_model('efficientnet_b4', pretrained=pretrained, num_classes=0)
        rgb_features = self.rgb_backbone.num_features
        self.fft_branch = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Linear(rgb_features+128, 512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512,128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128,num_classes)
        )
    def forward(self, rgb, fft):
        r = self.rgb_backbone(rgb)
        f = self.fft_branch(fft).flatten(1)
        return self.classifier(torch.cat([r,f], dim=1))


# ── Load YOUR EXISTING TRAINED WEIGHTS ─────────────────────────────
model = DualBranchDeepfakeDetector(num_classes=CFG['num_classes'], pretrained=False).to(DEVICE)

prev_ckpt = torch.load(PREV_MODEL_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(prev_ckpt['model_state'])
print(f'✅ Loaded previous model — Val AUC: {prev_ckpt.get("val_auc", "N/A")}, Val Acc: {prev_ckpt.get("val_acc", "N/A")}')

# Unfreeze everything — we're fine-tuning the whole network gently
for p in model.parameters():
    p.requires_grad = True

total_p = sum(p.numel() for p in model.parameters())
print(f'   Total params: {total_p/1e6:.1f}M (all trainable for fine-tuning)')

## ✅ Cell 8 — Optimizer & Loss (gentle settings)

In [ ]:
from torch.optim.lr_scheduler import OneCycleLR

criterion = nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])

scheduler = OneCycleLR(
    optimizer, max_lr=CFG['lr'],
    steps_per_epoch=len(train_loader),
    epochs=CFG['epochs'],
    pct_start=0.15, final_div_factor=100,
)

print(f'✅ Optimizer ready — LR: {CFG["lr"]} (gentle fine-tuning rate)')

## ✅ Cell 9 — Checkpoint Utilities

In [ ]:
def save_checkpoint(epoch, model, optimizer, scheduler, best_auc, history):
    torch.save({
        'epoch': epoch, 'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(), 'best_auc': best_auc,
    }, LAST_CKPT_PATH)
    with open(HISTORY_FILE, 'w') as f:
        json.dump(history, f, indent=2)

def load_checkpoint():
    if os.path.exists(LAST_CKPT_PATH):
        ckpt = torch.load(LAST_CKPT_PATH, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt['model_state'])
        print(f'✅ Resumed from epoch {ckpt["epoch"]}, best AUC={ckpt["best_auc"]:.4f}')
        return ckpt['epoch']+1, ckpt['best_auc']
    return 0, 0.0

def load_history():
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE) as f: return json.load(f)
    return {'epoch':[], 'train_loss':[], 'val_loss':[], 'val_acc':[], 'val_auc':[], 'val_f1':[], 'lr':[]}

def mixup_data(x_rgb, x_fft, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    idx = torch.randperm(x_rgb.size(0)).to(DEVICE)
    return (lam*x_rgb+(1-lam)*x_rgb[idx], lam*x_fft+(1-lam)*x_fft[idx], y, y[idx], lam)

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam*criterion(pred, y_a) + (1-lam)*criterion(pred, y_b)

print('✅ Utilities ready')

## 🚀 Cell 10 — FINE-TUNING LOOP

In [ ]:
start_epoch, best_auc = load_checkpoint()
history = load_history()
patience_counter = 0
scaler = torch.cuda.amp.GradScaler()

# Use previous model's AUC as the baseline to beat
if best_auc == 0.0:
    best_auc = prev_ckpt.get('val_auc', 0.0)
    print(f'Baseline to beat: {best_auc:.4f}')

print(f'\n🚀 Fine-tuning from epoch {start_epoch+1}/{CFG["epochs"]}')
print('─'*70)

for epoch in range(start_epoch, CFG['epochs']):
    epoch_start = time.time()
    model.train()
    train_loss = 0.0

    pbar = tqdm(train_loader, desc=f'Ep {epoch+1:02d}/{CFG["epochs"]} [Train]', leave=False)
    for rgb, fft, labels in pbar:
        rgb, fft, labels = rgb.to(DEVICE), fft.to(DEVICE), labels.to(DEVICE)
        rgb_m, fft_m, y_a, y_b, lam = mixup_data(rgb, fft, labels, CFG['mixup_alpha'])

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            out  = model(rgb_m, fft_m)
            loss = mixup_criterion(criterion, out, y_a, y_b, lam)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    # Validation
    model.eval()
    val_loss = 0.0
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for rgb, fft, labels in tqdm(val_loader, desc=f'Ep {epoch+1:02d}/{CFG["epochs"]} [Val]  ', leave=False):
            rgb, fft, labels = rgb.to(DEVICE), fft.to(DEVICE), labels.to(DEVICE)
            with torch.cuda.amp.autocast():
                out  = model(rgb, fft)
                loss = criterion(out, labels)
            val_loss += loss.item()
            probs = torch.softmax(out, dim=1)[:,1].cpu().numpy()
            preds = out.argmax(dim=1).cpu().numpy()
            all_probs.extend(probs); all_preds.extend(preds); all_labels.extend(labels.cpu().numpy())

    avg_train = train_loss/len(train_loader)
    avg_val   = val_loss/len(val_loader)
    val_acc   = accuracy_score(all_labels, all_preds)
    val_auc   = roc_auc_score(all_labels, all_probs)
    val_f1    = f1_score(all_labels, all_preds)
    cur_lr    = optimizer.param_groups[0]['lr']
    elapsed   = time.time()-epoch_start

    history['epoch'].append(epoch+1)
    history['train_loss'].append(round(avg_train,5))
    history['val_loss'].append(round(avg_val,5))
    history['val_acc'].append(round(val_acc,5))
    history['val_auc'].append(round(val_auc,5))
    history['val_f1'].append(round(val_f1,5))
    history['lr'].append(cur_lr)

    star = '⭐' if val_auc > best_auc else '  '
    print(f'Ep {epoch+1:02d}/{CFG["epochs"]} {star} | Loss {avg_train:.4f}→{avg_val:.4f} | '
          f'Acc {val_acc:.4f} | AUC {val_auc:.4f} | F1 {val_f1:.4f} | LR {cur_lr:.2e} | {elapsed:.0f}s')

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save({
            'model_state': model.state_dict(), 'val_auc': val_auc, 'val_acc': val_acc,
            'epoch': epoch+1, 'cfg': CFG,
        }, BEST_MODEL_PATH)
        patience_counter = 0
        print(f'   ✅ New best model saved (AUC={val_auc:.4f})')
    else:
        patience_counter += 1

    save_checkpoint(epoch, model, optimizer, scheduler, best_auc, history)

    if patience_counter >= CFG['patience']:
        print(f'\n⚡ Early stopping at epoch {epoch+1}')
        break

print(f'\n✅ Fine-tuning complete! Best AUC = {best_auc:.4f}')
print(f'   (Previous model AUC was: {prev_ckpt.get("val_auc", "N/A")})')

## ✅ Cell 11 — Evaluate on Combined Test Set

In [ ]:
ckpt = torch.load(LAST_CKPT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])

export_path = os.path.join(MODEL_DIR, 'deepfake_detector_v2_final.pth')
torch.save({
    'model_state'  : ckpt['model_state'],
    'architecture' : 'DualBranchDeepfakeDetector',
    'backbone'     : 'efficientnet_b4',
    'num_classes'  : 2,
    'img_size'     : 224,
    'val_auc'      : history['val_auc'][-1],
    'val_acc'      : history['val_acc'][-1],
    'trained_epoch': ckpt['epoch']+1,
    'class_names'  : ['fake', 'real'],
    'training_data': '140k + Celeb-DF v2 (combined)',
}, export_path)

print(f'✅ v2 exported — Val AUC: {history["val_auc"][-1]:.4f}, Val Acc: {history["val_acc"][-1]*100:.2f}%')

In [ ]:
model_v1 = DualBranchDeepfakeDetector(num_classes=2, pretrained=False).to(DEVICE)
v1_ckpt = torch.load(PREV_MODEL_PATH, map_location=DEVICE, weights_only=False)
model_v1.load_state_dict(v1_ckpt['model_state'])
model_v1.eval()

all_probs_v1, all_preds_v1, all_labels_v1 = [], [], []
with torch.no_grad():
    for rgb, fft, labels in tqdm(test_loader, desc='Testing v1 on combined set'):
        rgb, fft = rgb.to(DEVICE), fft.to(DEVICE)
        out = model_v1(rgb, fft)
        probs = torch.softmax(out, dim=1)[:,1].cpu().numpy()
        preds = out.argmax(dim=1).cpu().numpy()
        all_probs_v1.extend(probs); all_preds_v1.extend(preds); all_labels_v1.extend(labels.numpy())

v1_auc = roc_auc_score(all_labels_v1, all_probs_v1)
v1_acc = accuracy_score(all_labels_v1, all_preds_v1)
print(f'v1 on combined test set — AUC: {v1_auc:.4f}, Acc: {v1_acc*100:.2f}%')

In [ ]:
BEST_MODEL_PATH = os.path.join(MODEL_DIR, 'deepfake_detector_v2_final.pth')
ckpt = torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'✅ Loaded best fine-tuned model — epoch {ckpt["trained_epoch"]}, AUC={ckpt["val_auc"]:.4f}')

all_probs, all_preds, all_labels = [], [], []
with torch.no_grad():
    for rgb, fft, labels in tqdm(test_loader, desc='Testing'):
        rgb, fft = rgb.to(DEVICE), fft.to(DEVICE)
        with torch.cuda.amp.autocast():
            out = model(rgb, fft)
        probs = torch.softmax(out, dim=1)[:,1].cpu().numpy()
        preds = out.argmax(dim=1).cpu().numpy()
        all_probs.extend(probs); all_preds.extend(preds); all_labels.extend(labels.numpy())

test_auc = roc_auc_score(all_labels, all_probs)
test_acc = accuracy_score(all_labels, all_preds)
test_f1  = f1_score(all_labels, all_preds)

print('\n'+'='*50)
print('   FINE-TUNED MODEL — COMBINED TEST RESULTS')
print('='*50)
print(f'  Accuracy : {test_acc*100:.2f}%')
print(f'  AUC-ROC  : {test_auc:.4f}')
print(f'  F1 Score : {test_f1:.4f}')
print('='*50)
print()
print(classification_report(all_labels, all_preds, target_names=['Fake','Real']))

fig, axes = plt.subplots(1,2, figsize=(12,4))
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Fake','Real'], yticklabels=['Fake','Real'], ax=axes[0])
axes[0].set_title('Confusion Matrix'); axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

fpr, tpr, _ = roc_curve(all_labels, all_probs)
axes[1].plot(fpr, tpr, color='#534AB7', lw=2, label=f'AUC={test_auc:.3f}')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#534AB7')
axes[1].plot([0,1],[0,1],'--', color='gray')
axes[1].set_title('ROC Curve'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'finetune_evaluation.png'), dpi=150, bbox_inches='tight')
plt.show()

## ✅ Cell 12 — Export & Download Fine-tuned Model

In [ ]:
print('='*50)
print('  v1 (original) vs v2 (fine-tuned) — combined test set')
print('='*50)
print(f'  v1 — Acc: {v1_acc*100:.2f}% | AUC: {v1_auc:.4f}')
print(f'  v2 — Acc: {test_acc*100:.2f}% | AUC: {test_auc:.4f}')
print('='*50)

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/v2_outputs', 'zip', '/kaggle/working')

from IPython.display import FileLink
FileLink('v2_outputs.zip')

In [ ]:
export_path = os.path.join(MODEL_DIR, 'deepfake_detector_v2_final.pth')
torch.save({
    'model_state'  : ckpt['model_state'],
    'architecture' : 'DualBranchDeepfakeDetector',
    'backbone'     : 'efficientnet_b4',
    'num_classes'  : 2,
    'img_size'     : 224,
    'val_auc'      : ckpt['val_auc'],
    'val_acc'      : ckpt['val_acc'],
    'trained_epoch': ckpt['trained_epoch'],
    'class_names'  : ['fake', 'real'],
    'training_data': '140k Real/Fake Faces + Celeb-DF v2 (combined)',
}, export_path)

print(f'✅ Exported: {export_path}')
print(f'   Size: {os.path.getsize(export_path)/1e6:.1f} MB')
print(f'   Val AUC: {ckpt["val_auc"]:.4f}  |  Val Acc: {ckpt["val_acc"]*100:.2f}%')

# Download link
from IPython.display import FileLink
FileLink('models/deepfake_detector_v2_final.pth')